# EdgeCoRAG v1.10.1 Results

## tl;dr

- 全部 161 个任务、21927 次问答已完成，原始结果已备份。
- 冻结测试集上，五模型的 S2 均优于 S1；检索证据 S1 相比 S0 没有一致提升。
- 正式准确率与性能重复严格分开；格式失败始终计错。

## Context & Methods

本笔记本是正式结果报告的可复现计算附件，只读取 Mac 备份，不连接服务器、不运行模型。

### Key Assumptions

- S0：仅问题图像；S1：发布包中的 MRAG top-3 检索证据；S2：实际 GT 证据（oracle，对部署能力不作同等解释）。
- internal_holdout 每条件 150 题；frozen_test 每条件 1033 题；Incomplete 保留在全分母。
- 严格解析、64 token、BF16、无量化参数保持原样。
- 配对比较复用批准版 analyze_batch.py：同模型、同问题集、按场景分层 bootstrap，2000 次，seed=42。区间为描述性、未校正多重比较，未使用不可得的来源聚类结构。
- S3 仅 Gemma4，留出集 2 题、冻结集 29 题，按预先冻结的 AI 图像辅助审阅子集与 S1 做同题配对；不是人工复核或全基准结论。
- 冻结集存在已披露的审计暴露，不能称为新盲测或外部验证。
- 性能仅 RTX4090 上离线证据生成：不含检索、不含加载、不含 warmup；主条件每组 20 题 × 5 次，S3 每组 5 题 × 5 次。

### 1. Setup

In [ ]:
from pathlib import Path
import json, sys, os, tempfile, pandas as pd
import analyze_results as analysis
analysis.BACKUP = Path(os.environ['EDGECORAG_BACKUP_ROOT']).resolve()
analysis.OUT = Path(tempfile.mkdtemp(prefix='edgecorag-analysis-'))
print('Python', sys.version.split()[0])
print('Output:', analysis.OUT)


## Data

### 2. Validate and rescore

核对源码/矩阵/答案/输入哈希、完整性、复合主键、单次尝试、BF16、图片数量、warmup 分离，并用原始答案独立重算 76 份非性能评分。

In [ ]:
tasks, audit = analysis.load_and_validate()
print(json.dumps(audit, ensure_ascii=False, indent=2))

## Results

### 3. Calculate paired evidence differences

In [ ]:
results = analysis.compute(tasks, resamples=2000, seed=42)
analysis.save_tables(results, audit)
q = pd.DataFrame(results['quality'])
frozen = q[(q['split']=='frozen_test') & q['condition'].isin(['S0','S1','S2'])]
display((frozen.pivot(index='model', columns='condition', values='accuracy')*100).round(2))

### 4. Inspect uncertainty

以下区间保持同题配对，并非不同模型之间的因果比较。

In [ ]:
paired = pd.DataFrame(results['paired_comparisons'])
display(paired[(paired['split']=='frozen_test') & paired['comparison'].isin(['S1 − S0','S2 − S1'])][['model','comparison','n','delta_pp','ci_low_pp','ci_high_pp','improvements','regressions']].round(3))

### 5. Inspect format failures and limited S3 coverage

留出集 S3 只有2题；按场景分层可能产生退化区间，该区间不能表示高置信度，本报告不作统计解释。

In [ ]:
fail = pd.DataFrame(results['format_failures'])
display(fail.groupby(['stage','model']).size().rename('format_failures').to_frame())
display(pd.DataFrame(results['s3_subsets']).drop(columns=['model_key','ci_low_pp','ci_high_pp']).round(4))

### 6. Summarize performance

P95 仅为 100 次重复观测的描述统计，不等于 100 道独立题目。S3 每组仅 25 次，不报告 P95。显存为进程分配器峰值，不是设备级采样峰值。

In [ ]:
perf = pd.DataFrame(results['performance'])
display(perf[['model','condition','n_observations','unique_questions','end_to_end_p50_ms','end_to_end_p95_ms','format_failures']].round(2))
display(pd.DataFrame(results['memory']).drop(columns=['hardware_id','model_key']).round(3))

### 7. Export research figures

所有图表从已核对表生成，PNG/SVG/PDF 三种格式；主报告的交互图另由规范化聚合快照提供。

In [ ]:
paths = analysis.make_figures(results)
from IPython.display import Image, display
for path in paths:
    display(Image(filename=path, width=1000))

### 8. Reconcile raw-record SQL sources

图表与精确查表还由本机 SQLite 对原始问答和答案重新聚合；配对区间复用前述批准版 bootstrap 输出，不另造一套统计方法。P50/P95 的 SQL 插值与 Python 分位数逐项核对。

In [ ]:
sql_charts = analysis.build_sql_sources(tasks)
sql_tables = analysis.build_report_queries(tasks, results)
for row in sql_charts['frozen_quality']:
    expected = next(v for v in results['quality'] if (v['model_key'],v['split'],v['condition'])==(row['model_key'],row['split'],row['condition']))
    assert abs(row['accuracy']-expected['accuracy']) < 1e-12
for row in sql_tables['performance_lookup']:
    expected = next(v for v in results['performance'] if v['model_key']==row['model_key'] and v['condition']==row['condition'])
    assert abs(row['p50']-expected['end_to_end_p50_ms']) < 1e-8
    assert (row['p95'] is None)==(expected['end_to_end_p95_ms'] is None)
    if row['p95'] is not None:
        assert abs(row['p95']-expected['end_to_end_p95_ms']) < 1e-8
print('SQL/Python reconciliation passed; no server or inference involved.')

## Takeaways

- 可以使用当前结果整理报告，无需重新启动 GPU。
- 主叙述应区分检索证据的混合效果和 GT 证据的 oracle 上界，不能把 S2 当作真实检索部署结果。
- S3 是小规模、带审阅选择限制的补充结果。
- 单卡固定工作负载的性能数据不支持真实边缘设备、检索全链路或普遍稳定性的结论。
- 不在看到冻结集结果后改提示词、token 上限、精度或挑选子集再重跑。